# Tabular 03 · Preliminary tabular models and the superseded holdout result

**Objective:** show how the 24-feature tabular Random Forest was first chosen and tuned, and why its 0.861 holdout macro F1 is **not** the project's performance estimate.

**Derived from:** `notebooks/04_feature_preparation.ipynb` (feature-group ablation), `notebooks/05_model_selection_and_evaluation.ipynb` (model comparison, tuning, holdout).
**Design:** V0, a stratified 80/20 **row-level** holdout (828 / 208) with 5-fold CV on the 828 for comparison and tuning. **Runtime:** about 2 minutes.

> **SUPERSEDED.** These folds ignore donors: fellow eyes of one donor can sit on both sides of a split, and the holdout informed later decisions. All results here are preliminary. The donor-grouped results start in Hybrid 02. Notebook 05's SHAP summary (computed on training tissues) is also superseded, by Hybrid 05, and is not repeated.

Set up paths and load the labelled dataset with the saved V0 split (the helper checks the split against notebook 04).

In [1]:
import sys
import warnings
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
sys.path[:0] = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

import cleaned_setup as cs
import modeling_setup as ms

warnings.filterwarnings("ignore", category=UserWarning)
labelled = ms.load_labelled_data()
train_df, test_df = ms.get_holdout_split(labelled)
X_train, y_train = train_df[ms.ALL_FEATURES], train_df[ms.TARGET]
X_test, y_test = test_df[ms.ALL_FEATURES], test_df[ms.TARGET]
print(f"Training {len(train_df)} {y_train.value_counts().to_dict()} | holdout {len(test_df)} {y_test.value_counts().to_dict()}")

Training 828 {'A+': 436, 'A': 255, 'B': 137} | holdout 208 {'A+': 109, 'A': 64, 'B': 35}


## 1. Preprocessing that keeps missing values distinguishable

Konan values: median imputation (fitted on training data) and scaling. Slit-lamp findings: a missing value becomes its own category (`-1` for binary flags, `__MISSING__` for categorical ones) before one-hot encoding, so "not recorded" is never read as "No".

In [2]:
cs.show_source(ms.make_preprocessor)

def make_preprocessor(
    numeric=NUMERIC_FEATURES,
    binary=CLINICAL_BINARY_FEATURES,
    categorical=CLINICAL_CATEGORICAL_FEATURES,
    scale_numeric=True,
):
    """Build the 04 preprocessing for any subset of feature groups.

    Pass an empty list to drop a group, e.g. make_preprocessor(binary=[],
    categorical=[]) for numeric-only models. Missing clinical values are
    kept as their own category, as in 04.
    """
    transformers = []

    if numeric:
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            steps.append(("scaler", StandardScaler()))
        transformers.append(("numeric", Pipeline(steps), list(numeric)))

    if binary:
        transformers.append((
            "clinical_binary",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            list(binary),
        ))

    if cat

## 2. Which feature groups carry the signal? (notebook 04, preliminary)

Random Forest with 5-fold stratified CV on the 828 training tissues, for each feature group.

In [3]:
MORPHOLOGY = [f for f in ms.NUMERIC_FEATURES if f != cs.ECD]
GROUPS = {
    "ECD only": [cs.ECD],
    "ECD + HEX": [cs.ECD, "HEX_percent"],
    "Morphology without ECD": MORPHOLOGY,
    "ECD + all morphology (6 Konan)": ms.NUMERIC_FEATURES,
    "Slit-lamp findings only (18)": ms.CLINICAL_FEATURES,
    "All 24 features": ms.ALL_FEATURES,
}


def group_pipeline(features):
    return Pipeline([
        ("preprocessor", ms.make_preprocessor(
            numeric=[f for f in features if f in ms.NUMERIC_FEATURES],
            binary=[f for f in features if f in ms.CLINICAL_BINARY_FEATURES],
            categorical=[f for f in features if f in ms.CLINICAL_CATEGORICAL_FEATURES])),
        ("classifier", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)),
    ])


SCORING = {"Accuracy": "accuracy", "Balanced accuracy": "balanced_accuracy", "Macro F1": "f1_macro"}
ablation = {}
for name, features in GROUPS.items():
    scores = cross_validate(group_pipeline(features), X_train[features], y_train, cv=ms.make_cv(), scoring=SCORING, n_jobs=-1)
    ablation[name] = {m: scores[f"test_{m}"].mean() for m in SCORING} | {"Macro F1 SD": scores["test_Macro F1"].std()}
pd.DataFrame(ablation).T.round(3)

,Accuracy,Balanced accuracy,Macro F1,Macro F1 SD
ECD only,0.772,0.772,0.772,0.027
ECD + HEX,0.810,0.805,0.814,0.025
Morphology without ECD,0.778,0.761,0.767,0.042
ECD + all morphology (6 Konan),0.833,0.820,0.833,0.030
Slit-lamp findings only (18),0.337,0.372,0.315,0.050
All 24 features,0.838,0.827,0.834,0.026


## 3. Model family comparison (notebook 05, preliminary)

Four classifiers on all 24 features, same preprocessing and folds.

In [4]:
FAMILIES = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
    "SVM (RBF)": SVC(class_weight="balanced", random_state=42),
}
family_scores = {}
for name, model in FAMILIES.items():
    scores = cross_validate(Pipeline([("preprocessor", ms.make_preprocessor()), ("classifier", model)]),
                            X_train, y_train, cv=ms.make_cv(), scoring=SCORING, n_jobs=-1, error_score="raise")
    family_scores[name] = {m: scores[f"test_{m}"].mean() for m in SCORING} | {"Macro F1 SD": scores["test_Macro F1"].std()}
pd.DataFrame(family_scores).T.sort_values("Macro F1", ascending=False).round(3)

,Accuracy,Balanced accuracy,Macro F1,Macro F1 SD
Random Forest,0.838,0.827,0.834,0.026
SVM (RBF),0.794,0.781,0.789,0.028
Extra Trees,0.778,0.742,0.754,0.031
Logistic regression,0.729,0.731,0.719,0.014


## 4. Random Forest tuning (notebook 05; 48 settings × 5 folds, ≈1 min)

In [5]:
grid = GridSearchCV(
    Pipeline([("preprocessor", ms.make_preprocessor()),
              ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1))]),
    param_grid={
        "classifier__n_estimators": [300, 500],
        "classifier__max_depth": [None, 15, 25],
        "classifier__min_samples_split": [2, 5],
        "classifier__min_samples_leaf": [1, 2],
        "classifier__max_features": ["sqrt", 0.8],
    },
    scoring="f1_macro", cv=ms.make_cv(), n_jobs=-1, refit=True,
)
grid.fit(X_train, y_train)
print(f"Best CV macro F1: {grid.best_score_:.4f}")
print("Best settings:", {k.replace("classifier__", ""): v for k, v in grid.best_params_.items()})
print("These are the 'notebook 05 settings' reused unchanged in notebooks 06–12:",
      {k: v for k, v in cs.NB05_RF_PARAMS.items() if k in ("n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features")})

Best CV macro F1: 0.8535
Best settings: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
These are the 'notebook 05 settings' reused unchanged in notebooks 06–12: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


## 5. The holdout result: 0.861 macro F1 (SUPERSEDED)

The tuned forest on the 208 held-out tissues.

In [6]:
best_rf = grid.best_estimator_
pred = best_rf.predict(X_test)
print(f"Holdout macro F1 {f1_score(y_test, pred, average='macro'):.3f} | balanced accuracy {balanced_accuracy_score(y_test, pred):.3f} | accuracy {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, labels=cs.CLASSES, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(y_test, pred, labels=cs.CLASSES), index=[f"Recorded {c}" for c in cs.CLASSES], columns=[f"Predicted {c}" for c in cs.CLASSES])

Holdout macro F1 0.861 | balanced accuracy 0.857 | accuracy 0.856
              precision    recall  f1-score   support

          A+      0.912     0.853     0.882       109
           A      0.733     0.859     0.791        64
           B      0.968     0.857     0.909        35

    accuracy                          0.856       208
   macro avg      0.871     0.857     0.861       208
weighted avg      0.866     0.856     0.858       208



,Predicted A+,Predicted A,Predicted B
Recorded A+,93,16,0
Recorded A,8,55,1
Recorded B,1,4,30


## 6. Why this estimate was optimistic

Infer donors (Hybrid 02) and count holdout tissues whose fellow eye was in the training set; fellow eyes share a grade far more often than chance (77.9% vs 39.9%).

In [7]:
labelled["Donor_Group"] = ms.get_donor_groups(labelled)
donor_of = labelled.set_index("Tissue_ID")["Donor_Group"]
train_donors = set(donor_of[train_df["Tissue_ID"]])
test_with_fellow = test_df["Tissue_ID"].map(donor_of).isin(train_donors)
sizes = labelled["Donor_Group"].value_counts()
print(f"Fellow-eye pairs: {(sizes == 2).sum()}; pairs split between training and holdout: {test_with_fellow.sum()}")
print(f"Holdout tissues whose fellow eye was used for training: {test_with_fellow.sum()} of {len(test_df)} ({test_with_fellow.mean():.0%})")

Fellow-eye pairs: 258; pairs split between training and holdout: 78
Holdout tissues whose fellow eye was used for training: 78 of 208 (38%)


## Conclusion (preliminary)

- ECD alone already reaches ≈0.77 macro F1; adding the other Konan measurements raises it to ≈0.83, slit-lamp findings alone are weak (≈0.31), and all 24 features give ≈0.83 in this row-level CV.
- The Random Forest ranked first among four families and was tuned to `max_depth=15, min_samples_split=5, max_features="sqrt"` (300 trees, balanced class weights).
- The 0.861 holdout macro F1 is **superseded**. 78 of 258 donor pairs were split across training and holdout, and the holdout informed later decisions. Grouping by donor lowers the same model to 0.844 (Hybrid 02). Restricting to non-2023 tissues and tuning with nested donor-grouped CV gives 0.805 for the tabular model (Hybrid 06). Each of those numbers uses a different cohort or design, so they are not directly comparable with this one.

**Limitations:** row-level folds, a single reused holdout, and tuning on tabular features only (these settings later under-tuned the tabular arm by ≈0.05 macro F1, Hybrid 06).